In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display
import celloracle as co

print("Python:", sys.executable)
print("CellOracle:", co.__version__)


/home/ran3ds/.conda/envs/celloracle_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
which: no R in (/apps/software/standard/core/jupyterlab/4.5.6-py3.13/sbin:/apps/software/standard/core/jupyterlab/4.5.6-py3.13/bin:/opt/mam/9.1.2/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin:/opt/slurm/current/bin:/opt/rci/bin:/share/resources/HPCtools:/opt/mam/current/bin:/opt/apptainer/current/bin)


Python: /home/ran3ds/.conda/envs/celloracle_env/bin/python
CellOracle: 0.20.0


In [2]:
practicedata = sc.read_h5ad("Stage18_Data.h5ad")

/home/ran3ds/.conda/envs/celloracle_env/lib/python3.10/site-packages/anndata/compat/__init__.py:311: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


In [3]:
practicedata.obs["seurat_clusters"] = (
    practicedata.obs["seurat_clusters"].astype(str)
)
wt_adata = practicedata[
    (practicedata.obs["orig.ident"] == "pax6_WT") &
    (practicedata.obs["seurat_clusters"].isin(["6", "9"]))
].copy()

mutant_adata = practicedata[
    (practicedata.obs["orig.ident"] == "pax6_Mutant") &
    (practicedata.obs["seurat_clusters"].isin(["6", "9"]))
].copy()
print("WT:", wt_adata.shape)
print("Mutant:", mutant_adata.shape)

WT: (362, 22806)
Mutant: (450, 22806)


In [4]:
#this is for scaling an nonegative 
wt_adata.X = wt_adata.raw[:, wt_adata.var_names].X.copy()
mutant_adata.X = mutant_adata.raw[:, mutant_adata.var_names].X.copy()

In [5]:
base_GRN = co.data.load_xenopus_tropicalis_promoter_base_GRN(
    version="xenTro3_CisBPv2_fpr2"
)
display(base_GRN.head())


Loading prebuilt promoter base-GRN. Version: xenTro3_CisBPv2_fpr2


,peak_id,gene_short_name,AAI21521,AAI57209,AAI57537,AAI57586,AAI58894,AAI60957,AAI60967,AAI66068,...,zic4,zic5,znf143,znf238.2,znf276,znf281,znf534,znf652,znf740,zzz3
0,GL172637_1045291_1046391,izumo1r,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,GL172637_1056948_1058048,mast2,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,GL172637_1268461_1269561,pik3r3,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,GL172637_1717932_1719032,tspan1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,GL172637_1786405_1787505,pomgnt1,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Combine WT and mutant/delete genes so cell oracle can run 
pooled = sc.concat([wt_adata, mutant_adata]).copy()

# Remove genes detected in fewer than 10 cells
sc.pp.filter_genes(pooled, min_cells=10)

# Select approximately 2000 variable genes
sc.pp.highly_variable_genes(
    pooled,
    flavor="seurat",
    n_top_genes=2000,
    subset=True
)

# Keep the selected genes in both datasets
wt_adata = wt_adata[:, pooled.var_names].copy()
mutant_adata = mutant_adata[:, pooled.var_names].copy()

print("WT:", wt_adata.shape)
print("Mutant:", mutant_adata.shape)

WT: (362, 2000)
Mutant: (450, 2000)


In [8]:
# Store cluster labels as categories for CellOracle
wt_adata.obs["seurat_clusters"] = wt_adata.obs["seurat_clusters"].astype("category")

mutant_adata.obs["seurat_clusters"] = mutant_adata.obs["seurat_clusters"].astype("category")

# Organize the datasets for the following cells
datasets = {
    "WT": wt_adata,
    "Mutant": mutant_adata
}

print("WT:", wt_adata.shape)
print("Mutant:", mutant_adata.shape)

WT: (362, 2000)
Mutant: (450, 2000)


In [9]:
# CellOracle: create an Oracle object and import the base GRN
# https://morris-lab.github.io/CellOracle.documentation/notebooks/04_Network_analysis/Network_analysis_with_Paul_etal_2015_data.html#2.-Make-Oracle-object

# CellOracle: import already log-normalized, unscaled expression
# https://morris-lab.github.io/CellOracle.documentation/modules/celloracle.html#celloracle.Oracle.import_anndata_as_normalized_count

oracles = {}

for label, adata in datasets.items():

    # Create a container for this dataset
    oracle = co.Oracle()

    # Import expression, cluster labels and UMAP coordinates
    oracle.import_anndata_as_normalized_count(
        adata=adata,
        cluster_column_name="seurat_clusters",
        embedding_name="X_umap",
        test_mode=True
    )

    # Set cluster colors using CellOracle's public method
    # https://morris-lab.github.io/CellOracle.documentation/modules/celloracle.html#celloracle.Oracle.update_cluster_colors
    oracle.update_cluster_colors(
        palette={"6": "#4477AA", "9": "#EE6677"}
    )

    # Import possible TF-to-gene connections
    oracle.import_TF_data(TF_info_matrix=base_GRN)

    # Store the WT or mutant object
    oracles[label] = oracle

    print(label, "import complete")

... storing 'orig.ident' as categorical
... storing 'orig.ident' as categorical


WT import complete
Mutant import complete


In [11]:
# Based on CellOracle's PCA and KNN tutorial:
# https://morris-lab.github.io/CellOracle.documentation/notebooks/04_Network_analysis/Network_analysis_with_Paul_etal_2015_data.html#3.-KNN-imputation

N_PCS = 20
for label, oracle in oracles.items():
    oracle.perform_PCA()
    cumulative_variance = np.cumsum(oracle.pca.explained_variance_ratio_)
    plt.figure(figsize=(5, 3))
    plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance)
    plt.axvline(N_PCS, color="black", linestyle="--")
    plt.xlabel("Number of PCs")
    plt.ylabel("Cumulative explained variance")
    plt.title(label)
    plt.xlim(1, min(50, len(cumulative_variance)))
    plt.tight_layout()
    plt.show()
display

<function IPython.core.display_functions.display(*objs, include=None, exclude=None, metadata=None, transient=None, display_id=None, raw=False, clear=False, **kwargs)>

In [12]:
#https://morris-lab.github.io/CellOracle.documentation/notebooks/04_Network_analysis/Network_analysis_with_Paul_etal_2015_data.html#3.-KNN-imputation
for label, oracle in oracles.items():

    # Count the cells in this dataset
    n_cells = oracle.adata.shape[0]

    # Use 2.5% of the cells as neighbors, as in the tutorial
    k = int(0.025 * n_cells)

    # Smooth expression using nearby cells in PCA space
    oracle.knn_imputation(
        n_pca_dims=N_PCS,
        k=k,
        balanced=True,
        b_sight=k * 8,
        b_maxl=k * 4,
        n_jobs=4
    )

    print(label, "KNN complete. Neighbors:", k)

WT KNN complete. Neighbors: 9
Mutant KNN complete. Neighbors: 11


In [13]:
networks = {}
for label, oracle in oracles.items():
    networks[label] = oracle.get_links(
        cluster_name_for_GRN_unit="seurat_clusters",
        alpha=10,
        bagging_number=20,
        model_method="bagging_ridge",
        verbose_level=10,
        n_jobs=4,
    )
    assert set(map(str, networks[label].links_dict)) == {"6", "9"}
    for cluster, edges in networks[label].links_dict.items():
        print(label, "cluster", cluster, "unfiltered edges:", len(edges))


  0%|          | 0/2 [00:00<?, ?it/s]

Inferring GRN for 6...


  0%|          | 0/795 [00:00<?, ?it/s]

Inferring GRN for 9...


  0%|          | 0/795 [00:00<?, ?it/s]

WT cluster 6 unfiltered edges: 35194
WT cluster 9 unfiltered edges: 35194


... (2 outputs left out)

  0%|          | 0/795 [00:00<?, ?it/s]

Inferring GRN for 9...


  0%|          | 0/795 [00:00<?, ?it/s]

Mutant cluster 6 unfiltered edges: 35194
Mutant cluster 9 unfiltered edges: 35194


In [14]:
#https://morris-lab.github.io/CellOracle.documentation/notebooks/04_Network_analysis/Network_analysis_with_Paul_etal_2015_data.html#6.-Network-preprocessing
# Keep up to 2000 strong connections per WT cluster
networks["WT"].filter_links(
    p=0.001,
    weight="coef_abs",
    threshold_number=2000
)

# Keep up to 2000 strong connections per mutant cluster
networks["Mutant"].filter_links(
    p=0.001,
    weight="coef_abs",
    threshold_number=2000
)

In [15]:
print("WT: cluster 6")
display(networks["WT"].filtered_links["6"].head(10))

# WT — cluster 9
print("WT: cluster 9")
display(networks["WT"].filtered_links["9"].head(10))

# Mutant — cluster 6
print("Mutant: cluster 6")
display(networks["Mutant"].filtered_links["6"].head(10))

# Mutant — cluster 9
print("Mutant: cluster 9")
display(networks["Mutant"].filtered_links["9"].head(10))


WT: cluster 6


,source,target,coef_mean,coef_abs,p,-logp
12902,foxd1,fgfr3,0.620164,0.620164,1.496728e-16,15.824857
12906,foxn4,fgfr3,0.337037,0.337037,1.491608e-06,5.826345
12976,foxd1,flrt3,0.283680,0.283680,3.059417e-14,13.514361
220,nr2f1,LOC100145533,-0.245184,0.245184,6.993834e-11,10.155285
796,e2f7,LOC733945,0.240684,0.240684,8.698855e-19,18.060538
207,msx2,LOC100145533,-0.236371,0.236371,3.356147e-12,11.474159
31891,e2f7,tk1,0.236078,0.236078,3.278935e-14,13.484267
687,rxra,LOC733712,-0.236062,0.236062,4.197457e-12,11.377014
238,vax2,LOC100145533,-0.235206,0.235206,5.401820e-09,8.267460
2300,rfx3,angptl1,0.234279,0.234279,2.772303e-14,13.557159


WT: cluster 9


,source,target,coef_mean,coef_abs,p,-logp
12902,foxd1,fgfr3,0.579766,0.579766,5.993717e-18,17.222304
12906,foxn4,fgfr3,0.352860,0.352860,5.232086e-09,8.281325
12976,foxd1,flrt3,0.349424,0.349424,5.161882e-15,14.287192
34070,sox5,vsx1,-0.345186,0.345186,1.831283e-16,15.737245
34073,rorb,vsx1,0.339747,0.339747,5.910474e-14,13.228378
796,e2f7,LOC733945,0.286716,0.286716,7.437613e-20,19.128566
11607,etv1,esam,0.278513,0.278513,1.666255e-14,13.778259
26016,rorb,rgma,-0.267671,0.267671,6.253242e-16,15.203895
28820,etv1,slc38a3,0.260741,0.260741,1.024633e-17,16.989432
22176,nr2f2,nrp2,-0.249384,0.249384,1.212042e-11,10.916482


Mutant: cluster 6


,source,target,coef_mean,coef_abs,p,-logp
12902,foxd1,fgfr3,0.410530,0.410530,9.534840e-18,17.020687
29824,rfx3,sox5,0.322028,0.322028,2.042080e-16,15.689927
18123,vax2,krt70,-0.279130,0.279130,5.656733e-13,12.247434
12906,foxn4,fgfr3,0.254819,0.254819,1.296623e-09,8.887186
34073,rorb,vsx1,0.246768,0.246768,4.206269e-13,12.376103
796,e2f7,LOC733945,0.243626,0.243626,2.527678e-19,18.597278
13619,sp7,fucolectin,0.239366,0.239366,2.841099e-08,7.546514
788,sox5,LOC733945,-0.235811,0.235811,7.013241e-15,14.154081
34047,msx1,vsx1,0.234243,0.234243,4.392756e-12,11.357263
25480,etv1,rasgef1a,0.227546,0.227546,1.682985e-13,12.773920


Mutant: cluster 9


,source,target,coef_mean,coef_abs,p,-logp
12902,foxd1,fgfr3,0.389942,0.389942,7.207213e-15,14.142233
34073,rorb,vsx1,0.253700,0.253700,1.669297e-11,10.777466
21956,nr2f1,nr2f2,0.240098,0.240098,1.310367e-11,10.882607
7787,lhx5,cldn6.1,0.220123,0.220123,5.795990e-06,5.236872
29824,rfx3,sox5,0.212736,0.212736,3.391798e-10,9.469570
25660,gli1,rbp1,0.206062,0.206062,1.240782e-12,11.906304
31308,vsx1,tbx3,0.194777,0.194777,8.690145e-17,16.060973
12906,foxn4,fgfr3,0.188962,0.188962,1.683644e-07,6.773750
25689,rorb,rbp1,0.186373,0.186373,3.179439e-12,11.497649
25986,e2f7,rfx3,-0.177681,0.177681,4.754579e-13,12.322888


In [16]:
#https://morris-lab.github.io/CellOracle.documentation/notebooks/04_Network_analysis/Network_analysis_with_Paul_etal_2015_data.html
for label, links in networks.items():
    if any(edges.empty for edges in links.filtered_links.values()):
        print(label, "has an empty filtered network; inspect it before calculating scores.")
        continue
    links.get_network_score()
    for cluster in ["6", "9"]:
        scores = links.merged_score
        selected = scores[scores["cluster"].astype(str) == cluster]
        print(label, "cluster", cluster, "top genes by outgoing connections")
        display(selected[selected["degree_out"] > 0].nlargest(10, "degree_out"))


WT cluster 6 top genes by outgoing connections


,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
sox5,83,0.234463,10,0.028249,73,0.206215,2386.0,0.928721,6
rorb,72,0.203390,0,0.000000,72,0.203390,0.0,0.702305,6
nr2f2,84,0.237288,13,0.036723,71,0.200565,4741.0,1.000000,6
nr2f1,77,0.217514,10,0.028249,67,0.189266,2801.0,0.910426,6
vax2,64,0.180791,0,0.000000,64,0.180791,0.0,0.521454,6
foxd1,65,0.183616,5,0.014124,60,0.169492,1237.0,0.659548,6
msx1,69,0.194915,10,0.028249,59,0.166667,1674.0,0.683694,6
lhx9,51,0.144068,0,0.000000,51,0.144068,0.0,0.386161,6
vax1,49,0.138418,0,0.000000,49,0.138418,0.0,0.325058,6
etv1,61,0.172316,13,0.036723,48,0.135593,2621.0,0.680224,6


WT cluster 9 top genes by outgoing connections


,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
rorb,99,0.255814,0,0.000000,99,0.255814,0.0,0.891516,9
nr2f2,104,0.268734,15,0.038760,89,0.229974,4006.0,1.000000,9
vax2,86,0.222222,0,0.000000,86,0.222222,0.0,0.542715,9
vsx1,96,0.248062,21,0.054264,75,0.193798,4085.0,0.966394,9
nr2f1,73,0.188630,7,0.018088,66,0.170543,2482.0,0.645739,9
vax1,64,0.165375,0,0.000000,64,0.165375,0.0,0.471238,9
foxd1,66,0.170543,5,0.012920,61,0.157623,1175.0,0.660000,9
lhx9,61,0.157623,0,0.000000,61,0.157623,0.0,0.344254,9
foxp4,71,0.183463,11,0.028424,60,0.155039,2329.0,0.636440,9
sox5,69,0.178295,11,0.028424,58,0.149871,1640.0,0.619798,9


Mutant cluster 6 top genes by outgoing connections


,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
vax2,127,0.335092,0,0.000000,127,0.335092,0.0,1.000000,6
rorb,101,0.266491,0,0.000000,101,0.266491,0.0,0.777557,6
nr2f2,89,0.234828,18,0.047493,71,0.187335,5196.0,0.778838,6
nr2f1,74,0.195251,7,0.018470,67,0.176781,2186.0,0.598230,6
vsx1,80,0.211082,19,0.050132,61,0.160950,2235.0,0.840099,6
lhx9,60,0.158311,0,0.000000,60,0.158311,0.0,0.321125,6
sox5,67,0.176781,9,0.023747,58,0.153034,1621.0,0.584116,6
barhl1,58,0.153034,0,0.000000,58,0.153034,0.0,0.394443,6
msx2,57,0.150396,0,0.000000,57,0.150396,0.0,0.329148,6
foxd1,55,0.145119,5,0.013193,50,0.131926,673.0,0.516811,6


Mutant cluster 9 top genes by outgoing connections


,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
rorb,75,0.206612,0,0.000000,75,0.206612,0.0,0.858368,9
vsx1,77,0.212121,21,0.057851,56,0.154270,2879.0,1.000000,9
vax2,56,0.154270,0,0.000000,56,0.154270,0.0,0.475201,9
lhx9,53,0.146006,0,0.000000,53,0.146006,0.0,0.360665,9
msx1,67,0.184573,14,0.038567,53,0.146006,2970.0,0.717867,9
nr2f2,68,0.187328,19,0.052342,49,0.134986,2634.0,0.873892,9
vax1,49,0.134986,0,0.000000,49,0.134986,0.0,0.287976,9
foxd1,53,0.146006,5,0.013774,48,0.132231,838.0,0.669517,9
sox5,56,0.154270,8,0.022039,48,0.132231,1289.0,0.603252,9
foxp4,62,0.170799,15,0.041322,47,0.129477,1656.0,0.649627,9


In [19]:
#see plots 
for label, links in networks.items():
    if any(edges.empty for edges in links.filtered_links.values()):
        continue
    print(label)
    links.plot_degree_distributions(plot_model=True)
    plt.show()


WT
6
9
Mutant
6
9


In [20]:
#save results 
output_dir = Path("CellOracle_student_results")
output_dir.mkdir(exist_ok=True)

for label, links in networks.items():
    oracles[label].to_hdf5(str(output_dir / f"{label}.celloracle.oracle"))
    links.to_hdf5(str(output_dir / f"{label}.celloracle.links"))
    for cluster in ["6", "9"]:
        links.links_dict[cluster].to_csv(
            output_dir / f"{label}_cluster{cluster}_unfiltered.csv", index=False
        )
        links.filtered_links[cluster].to_csv(
            output_dir / f"{label}_cluster{cluster}_filtered.csv", index=False
        )
    if links.merged_score is not None:
        links.merged_score.to_csv(output_dir / f"{label}_network_scores.csv")

pd.Series({"python": sys.version, "celloracle": co.__version__,
           "scanpy": sc.__version__, "requested_PCs": N_PCS}).to_csv(
    output_dir / "analysis_settings.csv", header=False
)
print("Results saved in:", output_dir.resolve())


Results saved in: /sfs/gpfs/tardis/home/ran3ds/CellOracle_student_results
